# Focus Area 4 — Agro-Ecological Zoning with PyAEZ

Countries: **Ethiopia · Tanzania · Eritrea · Djibouti · Somalia · Sudan**

**Fixes applied vs previous version:**
- `UtilityCalculations` → imports `UtilitiesCalc` (v2.2 rename)
- `setLocationTeperature` → replaced with `setMonthlyClimateData` (v2.2 API)
- `setMonthlyPrecipitation` / `setMonthlyPET` → folded into `setMonthlyClimateData`
- `getMoistureZone()` removed in v2.2 → derived from LGP manually
- Array transpose: PyAEZ v2.2 expects `(nrows, ncols, 12)` not `(12, nrows, ncols)`
- Somalia and Sudan added to all country configs

In [ ]:
# @title ### 0) Configuration {"display-mode":"form"}
# @markdown Options: 'Ethiopia' | 'Tanzania' | 'Eritrea' | 'Djibouti' | 'Somalia' | 'Sudan'

import os
import numpy as np
from datetime import date

COUNTRY = 'Ethiopia'

COUNTRY_BBOX = {
    'Ethiopia': [33.0,   3.0,  48.0,  15.0],
    'Tanzania': [29.0, -12.0,  41.0,  -1.0],
    'Eritrea':  [36.5,  12.5,  43.5,  18.0],
    'Djibouti': [41.5,  10.9,  43.5,  12.7],
    'Somalia':  [41.0,  -1.7,  51.5,  12.0],
    'Sudan':    [21.8,   8.7,  38.7,  22.2],
}

xmin, ymin, xmax, ymax = COUNTRY_BBOX[COUNTRY]
PRECIP_SOURCE = 'CHIRPS'
LT_START      = 2020
LT_END        = 2024

LOCAL_CACHE_DIR = f'./Datasets/{COUNTRY}'
RESULTS_DIR     = f'./outputs/FA4_AEZ_{COUNTRY}_{date.today().isoformat()}'
os.makedirs(LOCAL_CACHE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR,     exist_ok=True)

print(f'\u2705 CONFIG set.')
print(f'   Country  : {COUNTRY}')
print(f'   Bbox     : [{xmin}, {ymin}, {xmax}, {ymax}]')
print(f'   Precip   : {PRECIP_SOURCE}')
print(f'   Period   : {LT_START}\u20132024')
print(f'   Outputs  : {RESULTS_DIR}')

In [ ]:
# @title ### 1) GEE Authentication {"display-mode":"form"}

import ee, json, gdown

SA_KEY_FILE_ID = '181IKB3sJ3iUn6ZOZbg50htgH2JKcxFkT'
SA_KEY_PATH    = 'service_account_key.json'

print('\U0001f511 Downloading service account key...')
gdown.download(f'https://drive.google.com/uc?id={SA_KEY_FILE_ID}', SA_KEY_PATH, quiet=True)

if not os.path.exists(SA_KEY_PATH) or os.path.getsize(SA_KEY_PATH) == 0:
    raise RuntimeError('\u274c Key file download failed.')

with open(SA_KEY_PATH) as f:
    _key = json.load(f)

credentials = ee.ServiceAccountCredentials(email=_key['client_email'], key_file=SA_KEY_PATH)
ee.Initialize(credentials)
print(f'\u2705 GEE authenticated as: {_key["client_email"]}')

In [ ]:
# @title ### 2) Install dependencies & mount Shared Drive {"display-mode":"form"}
print('Installing dependencies...')
import subprocess
!apt-get install -y libgdal-dev gdal-bin > /dev/null 2>&1
_gdal_ver = subprocess.check_output(['gdal-config','--version']).decode().strip()
print(f'   GDAL version: {_gdal_ver}')
!pip install gdal[numpy]=={_gdal_ver} -q
!pip install pyaez==2.2 numba scipy -q
!pip install xarray rioxarray rasterio cartopy -q
print('\u2705 Dependencies installed.')

import os, json, shutil, importlib
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from datetime import date
from google.colab import drive

# ── PyAEZ v2.2 FIXED IMPORTS ─────────────────────────────────────────────────
PYAEZ_AVAILABLE     = False
ClimateRegime       = None
UtilityCalculations = None

try:
    from pyaez import ClimateRegime as _CR
    ClimateRegime   = _CR
    PYAEZ_AVAILABLE = True
    print('\u2705 PyAEZ ClimateRegime imported.')
    _cr_test = ClimateRegime.ClimateRegime()
    print(f'   Methods: {[m for m in dir(_cr_test) if not m.startswith("_")]}')
except ImportError as e:
    print(f'\u26a0\ufe0f  ClimateRegime import failed: {e}')

# v2.2 renamed UtilityCalculations.py -> UtilitiesCalc.py
for _uc_name in ['UtilitiesCalc', 'UtilityCalculations']:
    try:
        _mod = __import__('pyaez', fromlist=[_uc_name])
        UtilityCalculations = getattr(_mod, _uc_name)
        print(f'\u2705 PyAEZ {_uc_name} imported.')
        break
    except (ImportError, AttributeError):
        pass

if UtilityCalculations is None:
    try:
        import pyaez as _pyaez_pkg
        _pkg_dir = os.path.dirname(_pyaez_pkg.__file__)
        for _fname in ['UtilitiesCalc.py', 'UtilityCalculations.py']:
            _uc_path = os.path.join(_pkg_dir, _fname)
            if os.path.exists(_uc_path):
                _spec = importlib.util.spec_from_file_location(_fname[:-3], _uc_path)
                _uc_mod = importlib.util.module_from_spec(_spec)
                _spec.loader.exec_module(_uc_mod)
                UtilityCalculations = _uc_mod
                print(f'\u2705 {_fname} loaded via direct path.')
                break
    except Exception as _e:
        pass

if UtilityCalculations is None:
    print('\u26a0\ufe0f  UtilitiesCalc not found \u2014 Ra will use latitude approximation.')
if not PYAEZ_AVAILABLE:
    print('\u26a0\ufe0f  PyAEZ not available \u2014 manual FAO calculations will be used.')

# ── MOUNT SHARED DRIVE ────────────────────────────────────────────────────────
print('\n\U0001f4c2 Mounting Google Drive...')
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive', force_remount=False)
else:
    print('   Drive already mounted.')

SHARED_DRIVE_ROOT = '/content/drive/Shareddrives/NOAA-workshop2'
DATASETS_ROOT     = os.path.join(SHARED_DRIVE_ROOT, 'Datasets')
COUNTRY_DRIVE_DIR = os.path.join(DATASETS_ROOT, COUNTRY)

if not os.path.exists(SHARED_DRIVE_ROOT):
    print('\u274c Shared Drive not found.')
    DRIVE_CACHE_AVAILABLE = False
else:
    os.makedirs(COUNTRY_DRIVE_DIR, exist_ok=True)
    DRIVE_CACHE_AVAILABLE = True
    print(f'\u2705 Shared Drive ready: {COUNTRY_DRIVE_DIR}')

def try_load_from_drive(local_path, filename):
    if not DRIVE_CACHE_AVAILABLE: return False
    src = os.path.join(COUNTRY_DRIVE_DIR, filename)
    if os.path.exists(src):
        os.makedirs(os.path.dirname(local_path) or '.', exist_ok=True)
        shutil.copy(src, local_path)
        print(f'   \u2705 Loaded from Drive: {filename}')
        return True
    return False

def cache_to_drive(local_path, filename):
    if not DRIVE_CACHE_AVAILABLE: return
    dest = os.path.join(COUNTRY_DRIVE_DIR, filename)
    try:
        shutil.copy(local_path, dest)
        print(f'   \u2601\ufe0f  Cached to Drive: {filename}')
    except Exception as e:
        print(f'   \u26a0\ufe0f  Cache failed: {e}')

In [ ]:
# @title ### 3) Load ERA5 temperature & satellite precipitation {"display-mode":"form"}

def subset_to_bbox(ds, xmin, xmax, ymin, ymax):
    lat_dim = next((c for c in ds.coords if 'lat' in c.lower()), None)
    lon_dim = next((c for c in ds.coords if 'lon' in c.lower()), None)
    if not lat_dim or not lon_dim: return ds
    lv  = ds[lat_dim].values
    ls  = slice(ymax, ymin) if lv[0] > lv[-1] else slice(ymin, ymax)
    return ds.sel({lat_dim: ls, lon_dim: slice(xmin, xmax)})

# ERA5 temperature
_lt_filename = f'era5_temperature_monthly_{LT_START}_{LT_END}_{COUNTRY}.nc'
_lt_local    = os.path.join(LOCAL_CACHE_DIR, _lt_filename)
era5_lt      = None

print(f'\n\U0001f50d Loading ERA5 temperature: {_lt_filename}')
if os.path.exists(_lt_local):
    era5_lt = xr.open_dataset(_lt_local)
    print('   \u2705 Found locally.')
elif try_load_from_drive(_lt_local, _lt_filename):
    era5_lt = xr.open_dataset(_lt_local)

if era5_lt is None:
    print('   \u26a0\ufe0f  Not cached \u2014 pulling from GEE...')
    try:
        _region = ee.Geometry.BBox(xmin, ymin, xmax, ymax)
        _col = (
            ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')
            .filterDate(f'{LT_START}-01-01', f'{LT_END}-12-31')
            .select('temperature_2m')
        )
        _imgs = _col.toList(_col.size())
        _n    = _col.size().getInfo()
        print(f'   Pulling {_n} monthly images...')
        bands_list, times_list = [], []
        for i in range(_n):
            img   = ee.Image(_imgs.get(i))
            t_str = img.date().format('YYYY-MM-dd').getInfo()
            arr   = img.sampleRectangle(region=_region, defaultValue=0)
            vals  = np.array(arr.get('temperature_2m').getInfo(), dtype=float)
            vals  = np.where(vals > 150, vals - 273.15, vals)
            bands_list.append(vals)
            times_list.append(pd.Timestamp(t_str))
            if (i+1) % 12 == 0: print(f'   ... {i+1}/{_n}')
        _stack = np.stack(bands_list, axis=0)
        _lats  = np.linspace(ymax, ymin, bands_list[0].shape[0])
        _lons  = np.linspace(xmin, xmax, bands_list[0].shape[1])
        era5_lt = xr.Dataset(
            {'temperature_2m': (['time','lat','lon'], _stack)},
            coords={'time': times_list, 'lat': _lats, 'lon': _lons}
        )
        era5_lt.to_netcdf(_lt_local)
        cache_to_drive(_lt_local, _lt_filename)
        print(f'   \u2705 Pulled: {_stack.shape}')
    except Exception as e:
        print(f'   \u274c GEE pull failed: {e}')

if era5_lt is not None:
    era5_lt = subset_to_bbox(era5_lt, xmin, xmax, ymin, ymax)
    _da = era5_lt[list(era5_lt.data_vars)[0]]
    print(f'   Shape: {_da.shape}  | {str(_da.time.values[0])[:10]} \u2192 {str(_da.time.values[-1])[:10]}')

# Precipitation
precip_ds = None
_precip_filename = None
print(f'\n\U0001f50d Searching for {PRECIP_SOURCE} precipitation...')

if DRIVE_CACHE_AVAILABLE:
    _matching = [f for f in os.listdir(COUNTRY_DRIVE_DIR)
                 if PRECIP_SOURCE.upper() in f.upper() and f.endswith('.nc')]
    if _matching:
        _precip_filename = _matching[0]
        _precip_local    = os.path.join(LOCAL_CACHE_DIR, _precip_filename)
        if not os.path.exists(_precip_local):
            shutil.copy(os.path.join(COUNTRY_DRIVE_DIR, _precip_filename), _precip_local)
        precip_ds = xr.open_dataset(_precip_local)
        precip_ds = subset_to_bbox(precip_ds, xmin, xmax, ymin, ymax)
        print(f'   \u2705 Found: {_precip_filename}')
    else:
        print(f'   \u26a0\ufe0f  No {PRECIP_SOURCE} .nc in Shared Drive/{COUNTRY}/')
        if DRIVE_CACHE_AVAILABLE: print(f'   Available: {os.listdir(COUNTRY_DRIVE_DIR)}')

print(f'\n\U0001f4ca Summary:')
print(f'   ERA5 temp  : {"\u2705 spatial grid" if era5_lt is not None else "\u274c missing"}')
print(f'   Precip     : {"\u2705 " + str(_precip_filename) if precip_ds is not None else "\u274c missing"}')

In [ ]:
# @title ### 4) Prepare monthly climate arrays {"display-mode":"form"}

days_per_month = np.array([31,28,31,30,31,30,31,31,30,31,30,31])

# TEMPERATURE
print('Preparing temperature arrays...')
if era5_lt is not None:
    _da = era5_lt[list(era5_lt.data_vars)[0]]
    if float(_da.mean()) > 100: _da = _da - 273.15
    _clim_da = _da.resample(time='1ME').mean().groupby('time.month').mean('time')
    tavg_clim = _clim_da.values
    _lat_dim = next((c for c in era5_lt.coords if 'lat' in c.lower()), None)
    _lon_dim = next((c for c in era5_lt.coords if 'lon' in c.lower()), None)
    if _lat_dim and _lon_dim:
        map_lats    = era5_lt[_lat_dim].values
        map_lons    = era5_lt[_lon_dim].values
        HAS_SPATIAL = tavg_clim.ndim == 3
    else:
        HAS_SPATIAL = False
    tavg_monthly_1d = np.nanmean(tavg_clim, axis=(1,2)) if HAS_SPATIAL else tavg_clim.flatten()[:12]
    DIURNAL_RANGE   = 10.0
    tmin_clim       = tavg_clim - DIURNAL_RANGE/2
    tmax_clim       = tavg_clim + DIURNAL_RANGE/2
    tmin_monthly_1d = tavg_monthly_1d - DIURNAL_RANGE/2
    tmax_monthly_1d = tavg_monthly_1d + DIURNAL_RANGE/2
    print(f'   \u2705 Shape: {tavg_clim.shape}  HAS_SPATIAL={HAS_SPATIAL}')
    print(f'   \u2705 Tavg (\u00b0C): {np.round(tavg_monthly_1d,1)}')
else:
    print('\u26a0\ufe0f  ERA5 not available \u2014 using East Africa typical values.')
    tavg_monthly_1d = np.array([22,23,24,24,22,19,18,19,21,22,22,21],dtype=float)
    tmin_monthly_1d = tavg_monthly_1d - 5.0
    tmax_monthly_1d = tavg_monthly_1d + 5.0
    tavg_clim = tmin_clim = tmax_clim = None
    HAS_SPATIAL = False
    map_lats = map_lons = None

# PRECIPITATION
print('\nPreparing precipitation arrays...')
precip_clim = None
precip_monthly_1d = None
HAS_SPATIAL_PRECIP = False

if precip_ds is not None:
    _pda = precip_ds[list(precip_ds.data_vars)[0]]
    _p_clim_da = _pda.resample(time='1ME').sum().groupby('time.month').mean('time')
    precip_clim_raw = _p_clim_da.values
    n_months = precip_clim_raw.shape[0]
    if n_months < 12:
        print(f'   \u2139\ufe0f  Only {n_months} months available \u2014 filling missing with mean.')
        _known_months_idx = [int(m)-1 for m in _p_clim_da['month'].values]
        _shape = (12,) + precip_clim_raw.shape[1:]
        precip_clim_full = np.full(_shape, np.nan)
        for ki, i in zip(_known_months_idx, range(len(_known_months_idx))):
            precip_clim_full[ki] = precip_clim_raw[i]
        _known_mean = np.nanmean(precip_clim_full)
        _missing = [i for i in range(12) if np.all(np.isnan(precip_clim_full[i]))]
        for mi in _missing: precip_clim_full[mi] = _known_mean
        precip_clim = precip_clim_full
        print(f'   \u26a0\ufe0f  Months {[i+1 for i in _missing]} estimated at {_known_mean:.1f} mm')
    else:
        precip_clim = precip_clim_raw
    if precip_clim.ndim > 1:
        precip_monthly_1d = np.nanmean(precip_clim, axis=tuple(range(1,precip_clim.ndim)))
        HAS_SPATIAL_PRECIP = True
        if map_lats is None:
            _plat = next((c for c in precip_ds.coords if 'lat' in c.lower()), None)
            _plon = next((c for c in precip_ds.coords if 'lon' in c.lower()), None)
            if _plat and _plon:
                map_lats = precip_ds[_plat].values
                map_lons = precip_ds[_plon].values
    else:
        precip_monthly_1d = precip_clim
    print(f'   \u2705 Precip (mm): {np.round(precip_monthly_1d,1)}')
else:
    print('\u26a0\ufe0f  Precipitation not available \u2014 using typical values.')
    precip_monthly_1d = np.array([40,50,90,120,100,30,15,20,50,90,80,45],dtype=float)

# HARGREAVES PET
print('\nComputing monthly PET (Hargreaves)...')
lat_c = (ymin + ymax) / 2
Ra_monthly = None

if PYAEZ_AVAILABLE and UtilityCalculations is not None:
    try:
        if hasattr(UtilityCalculations, 'calcRa'):
            Ra_monthly = np.array([float(UtilityCalculations.calcRa(lat_c, m)) for m in range(1,13)])
        else:
            # Find any class in the module
            _cls_names = [x for x in dir(UtilityCalculations) if not x.startswith('_') and isinstance(getattr(UtilityCalculations, x), type)]
            if _cls_names:
                _uc = getattr(UtilityCalculations, _cls_names[0])()
                Ra_monthly = np.array([float(_uc.calcRa(lat_c, m)) for m in range(1,13)])
        if Ra_monthly is not None:
            print(f'   \u2705 Ra from PyAEZ: {np.round(Ra_monthly,1)}')
    except Exception as e:
        print(f'   \u26a0\ufe0f  PyAEZ Ra failed ({e}) \u2014 using approximation.')
        Ra_monthly = None

if Ra_monthly is None:
    Ra_monthly = 15.0 + 3.0 * np.cos(np.linspace(0, 2*np.pi, 12) + np.pi/6)
    print(f'   \u2705 Ra (latitude approx): {np.round(Ra_monthly,1)}')

pet_monthly_1d = (0.0023 * Ra_monthly * (tavg_monthly_1d + 17.8)
                  * np.sqrt(np.maximum(tmax_monthly_1d - tmin_monthly_1d, 0.1)))
pet_monthly_mm = pet_monthly_1d * days_per_month

if HAS_SPATIAL and tavg_clim is not None:
    pet_clim = np.zeros_like(tavg_clim)
    for m in range(12):
        _td = np.sqrt(np.maximum(tmax_clim[m] - tmin_clim[m], 0.1))
        pet_clim[m] = 0.0023 * Ra_monthly[m] * (tavg_clim[m]+17.8) * _td * days_per_month[m]
    print(f'   \u2705 Spatial PET: {pet_clim.shape}')
else:
    pet_clim = None

print(f'\n   PET (mm/month):    {np.round(pet_monthly_mm,0)}')
print(f'   Water balance:     {np.round(precip_monthly_1d - pet_monthly_mm,0)}')

# SPATIAL AEZ MAPS
print('\n\U0001f5fa\ufe0f  Computing spatial AEZ maps...')
if HAS_SPATIAL and tavg_clim is not None:
    _nrows_s, _ncols_s = tavg_clim.shape[1], tavg_clim.shape[2]
    _pet_spatial = pet_clim if pet_clim is not None else np.stack([
        0.0023*Ra_monthly[m]*(tavg_clim[m]+17.8)*np.sqrt(np.maximum(tmax_clim[m]-tmin_clim[m],0.1))*days_per_month[m]
        for m in range(12)], axis=0)

    if precip_clim is not None and precip_clim.ndim == 3:
        from scipy.interpolate import RegularGridInterpolator
        _plat = next((c for c in precip_ds.coords if 'lat' in c.lower()), None)
        _plon = next((c for c in precip_ds.coords if 'lon' in c.lower()), None)
        _p_lats = precip_ds[_plat].values if _plat else np.linspace(ymin,ymax,precip_clim.shape[1])
        _p_lons = precip_ds[_plon].values if _plon else np.linspace(xmin,xmax,precip_clim.shape[2])
        _prec_on_era5 = np.zeros((12,_nrows_s,_ncols_s))
        for m in range(12):
            try:
                _interp = RegularGridInterpolator(
                    (np.sort(_p_lats), np.sort(_p_lons)),
                    precip_clim[m] if _p_lats[0]<_p_lats[-1] else precip_clim[m][::-1],
                    method='linear', bounds_error=False, fill_value=np.nan)
                _gl, _glo = np.meshgrid(map_lats, map_lons, indexing='ij')
                _prec_on_era5[m] = _interp(np.stack([_gl,_glo],axis=-1))
            except Exception:
                _prec_on_era5[m] = precip_monthly_1d[m]
        precip_spatial = _prec_on_era5
    else:
        precip_spatial = precip_monthly_1d[:,np.newaxis,np.newaxis] * np.ones((12,_nrows_s,_ncols_s))

    lgp_map = ((precip_spatial >= 0.5*_pet_spatial).sum(axis=0)*30).astype(float)

    _t_min_g  = tavg_clim.min(axis=0)
    _t_mean_g = tavg_clim.mean(axis=0)
    _cold_m   = (tavg_clim < 5).sum(axis=0)
    thermal_map = np.zeros((_nrows_s,_ncols_s),dtype=float)
    thermal_map[_t_min_g  >= 20]                                             = 1
    thermal_map[(_t_min_g>=17)&(_t_min_g<20)&(_cold_m==0)]                 = 2
    thermal_map[(_t_mean_g>=15)&(_cold_m<3)&(thermal_map==0)]              = 3
    thermal_map[(_t_mean_g>=10)&(_cold_m<6)&(thermal_map==0)]              = 4
    thermal_map[(_t_mean_g>=5)&(thermal_map==0)]                            = 5
    thermal_map[(_t_mean_g>=0)&(thermal_map==0)]                            = 6
    thermal_map[thermal_map==0]                                              = 7

    moisture_map = np.zeros((_nrows_s,_ncols_s),dtype=float)
    moisture_map[lgp_map< 30]                           = 1
    moisture_map[(lgp_map>=30)&(lgp_map<60)]            = 2
    moisture_map[(lgp_map>=60)&(lgp_map<120)]           = 3
    moisture_map[(lgp_map>=120)&(lgp_map<180)]          = 4
    moisture_map[(lgp_map>=180)&(lgp_map<270)]          = 5
    moisture_map[lgp_map>=270]                           = 6
    aez_map = thermal_map*10 + moisture_map
    HAS_SPATIAL_MAPS = True
    print(f'   \u2705 LGP: min={lgp_map.min():.0f} max={lgp_map.max():.0f} mean={lgp_map.mean():.0f} days')
    print(f'   \u2705 Thermal zones : {np.unique(thermal_map).astype(int).tolist()}')
    print(f'   \u2705 Moisture zones: {np.unique(moisture_map).astype(int).tolist()}')
else:
    lgp_map=thermal_map=moisture_map=aez_map=precip_spatial=_pet_spatial=None
    HAS_SPATIAL_MAPS = False
    print('   \u26a0\ufe0f  No spatial grid available.')

In [ ]:
# @title ### 5) Module I \u2014 Climate Regime Analysis {"display-mode":"form"}
# @markdown Uses PyAEZ v2.2 setMonthlyClimateData() API.
# @markdown Falls back to manually computed maps if PyAEZ fails.

THERMAL_CLASSES = {
    1:'Tropics, lowland    (Tavg \u2265 20\u00b0C all months)',
    2:'Tropics, subtropics (Tavg 17\u201320\u00b0C, no frost)',
    3:'Subtropics, warm    (Tavg 10\u201320\u00b0C, < 3 cold months)',
    4:'Subtropics, cool    (Tavg 5\u201315\u00b0C, 3\u20136 cold months)',
    5:'Temperate, moderate (Tavg 0\u201310\u00b0C, frost present)',
    6:'Temperate, cool/cold (Tavg < 5\u00b0C in winter)',
    7:'Boreal / Arctic',
}
MOISTURE_CLASSES = {
    1:'Hyper-arid    (LGP < 30 days)',
    2:'Arid          (LGP 30\u201360 days)',
    3:'Semi-arid     (LGP 60\u2013120 days)',
    4:'Sub-humid     (LGP 120\u2013180 days)',
    5:'Humid         (LGP 180\u2013270 days)',
    6:'Per-humid     (LGP > 270 days)',
}

lgp_value = thermal_class = moisture_class = None

# ── PyAEZ v2.2 spatial run ────────────────────────────────────────────────────
if PYAEZ_AVAILABLE and HAS_SPATIAL_MAPS and tavg_clim is not None:
    try:
        _nrows_s, _ncols_s = tavg_clim.shape[1], tavg_clim.shape[2]
        _mask = np.ones((_nrows_s, _ncols_s), dtype=int)

        cr = ClimateRegime.ClimateRegime()
        cr.setStudyAreaMask(_mask, no_data_value=0)

        # FIX 1: v2.2 single method, FIX 2: transpose to (nrows,ncols,12)
        _T = lambda a: np.transpose(a, (1,2,0))
        cr.setMonthlyClimateData(
            min_temp      = _T(tmin_clim),
            max_temp      = _T(tmax_clim),
            mean_temp     = _T(tavg_clim),
            precipitation = _T(precip_spatial),
            short_rad     = None,
            wind_speed    = None,
        )

        cr.getThermalClimate()
        cr.getLGP(Sa=100, D50=0.5)
        cr.getThermalZone()

        _lgp_out      = np.array(cr.lgp, dtype=float)
        _thermal_out  = np.array(cr.thermal_climate, dtype=float)

        # FIX 3: getMoistureZone removed in v2.2 \u2014 derive from LGP
        def _lgp_to_mz(a):
            mz = np.zeros_like(a)
            mz[a<30]=1; mz[(a>=30)&(a<60)]=2; mz[(a>=60)&(a<120)]=3
            mz[(a>=120)&(a<180)]=4; mz[(a>=180)&(a<270)]=5; mz[a>=270]=6
            return mz

        _moisture_out = _lgp_to_mz(_lgp_out)

        lgp_map      = _lgp_out
        thermal_map  = _thermal_out
        moisture_map = _moisture_out
        aez_map      = thermal_map*10 + moisture_map

        lgp_value      = int(np.nanmean(lgp_map))
        thermal_class  = int(np.nanmedian(thermal_map))
        moisture_class = int(np.nanmedian(moisture_map))

        print('\u2705 PyAEZ ClimateRegime ran on FULL SPATIAL GRID.')
        print(f'   LGP        : min={lgp_map.min():.0f} max={lgp_map.max():.0f} mean={lgp_map.mean():.0f} days')
        print(f'   Thermal    : {np.unique(thermal_map[~np.isnan(thermal_map)]).astype(int).tolist()}')
        print(f'   Moisture   : {np.unique(moisture_map[~np.isnan(moisture_map)]).astype(int).tolist()}')

    except Exception as e:
        print(f'\u26a0\ufe0f  PyAEZ run failed: {e}')
        print('   Using manually computed maps from Cell 4.')

# Fallback scalars
if lgp_value is None:
    if HAS_SPATIAL_MAPS and lgp_map is not None:
        lgp_value      = int(np.nanmean(lgp_map))
        thermal_class  = int(np.nanmedian(thermal_map))
        moisture_class = int(np.nanmedian(moisture_map))
    else:
        def _lgp1d(p,pet): return int((p>=0.5*pet).sum()*30)
        def _therm(t):
            if t.min()>=20: return 1
            if t.min()>=17: return 2
            if t.mean()>=15: return 3
            if t.mean()>=10: return 4
            if t.mean()>=5:  return 5
            if t.mean()>=0:  return 6
            return 7
        def _moist(lgp):
            for t,c in [(30,1),(60,2),(120,3),(180,4),(270,5)]: 
                if lgp<t: return c
            return 6
        lgp_value      = _lgp1d(precip_monthly_1d, pet_monthly_mm)
        thermal_class  = _therm(tavg_monthly_1d)
        moisture_class = _moist(lgp_value)

tavg_annual   = tavg_monthly_1d.mean()
precip_annual = precip_monthly_1d.sum()

print('\n' + '\u2550'*60)
print(f'  CLIMATE REGIME RESULTS \u2014 {COUNTRY}')
print('\u2550'*60)
print(f'  Thermal class   : {thermal_class} \u2014 {THERMAL_CLASSES.get(thermal_class,"Unknown")}')
print(f'  LGP (mean)      : {lgp_value} days/year')
print(f'  Moisture zone   : {moisture_class} \u2014 {MOISTURE_CLASSES.get(moisture_class,"Unknown")}')
print(f'  Annual Tavg     : {tavg_annual:.1f}\u00b0C')
print(f'  Annual Precip   : {precip_annual:.0f} mm')
print(f'  Annual PET      : {pet_monthly_mm.sum():.0f} mm')
print(f'  Aridity index   : {precip_annual/pet_monthly_mm.sum():.2f} (P/PET)')
print(f'  Spatial maps    : {"\u2705 available" if HAS_SPATIAL_MAPS else "\u26a0\ufe0f  1D mode"}')
print('\u2550'*60)

In [ ]:
# @title ### 6) Spatial AEZ maps {"display-mode":"form"}

if not HAS_SPATIAL_MAPS or lgp_map is None:
    print('\u26a0\ufe0f  No spatial maps \u2014 run Cell 4 with spatial data first.')
else:
    _extent = [xmin, xmax, ymin, ymax]
    fig = plt.figure(figsize=(20,16))
    fig.suptitle(f'Agro-Ecological Zone Maps \u2014 {COUNTRY}', fontsize=14, fontweight='bold')

    ax1 = fig.add_subplot(2,2,1)
    _lc = ['#d73027','#f46d43','#fdae61','#a6d96a','#1a9850','#006837']
    _lc_map = mcolors.ListedColormap(_lc)
    _lc_norm = mcolors.BoundaryNorm([0,30,60,120,180,270,366], _lc_map.N)
    im1 = ax1.imshow(lgp_map, extent=_extent, cmap=_lc_map, norm=_lc_norm, origin='upper', aspect='auto')
    plt.colorbar(im1, ax=ax1, label='Days/year', shrink=0.8)
    ax1.set_title(f'LGP \u2014 Mean {lgp_map.mean():.0f} days/yr', fontsize=11)
    ax1.set_xlabel('Longitude'); ax1.set_ylabel('Latitude'); ax1.grid(alpha=0.3)

    ax2 = fig.add_subplot(2,2,2)
    _tc = ['#d73027','#fc8d59','#fee08b','#d9ef8b','#91cf60','#1a9850','#4575b4']
    _tm = int(thermal_map.max())
    _t_cmap = mcolors.ListedColormap(_tc[:_tm])
    _t_norm = mcolors.BoundaryNorm(np.arange(0.5,_tm+1.5), _t_cmap.N)
    im2 = ax2.imshow(thermal_map, extent=_extent, cmap=_t_cmap, norm=_t_norm, origin='upper', aspect='auto', interpolation='nearest')
    _tl = {1:'Tropics low',2:'Tropics/sub',3:'Subtropics warm',4:'Subtropics cool',5:'Temperate',6:'Temp cool',7:'Boreal'}
    ax2.legend(handles=[mpatches.Patch(color=_tc[int(v)-1],label=f'Class {int(v)}: {_tl.get(int(v),"?")}') for v in np.unique(thermal_map) if not np.isnan(v)], loc='lower left', fontsize=7, framealpha=0.8)
    ax2.set_title('Thermal Climate', fontsize=11); ax2.set_xlabel('Longitude'); ax2.set_ylabel('Latitude'); ax2.grid(alpha=0.3)

    ax3 = fig.add_subplot(2,2,3)
    _mc = ['#d73027','#f46d43','#fdae61','#fee090','#74add1','#313695']
    _mm = int(moisture_map.max())
    _m_cmap = mcolors.ListedColormap(_mc[:_mm])
    _m_norm = mcolors.BoundaryNorm(np.arange(0.5,_mm+1.5), _m_cmap.N)
    im3 = ax3.imshow(moisture_map, extent=_extent, cmap=_m_cmap, norm=_m_norm, origin='upper', aspect='auto', interpolation='nearest')
    _ml = {1:'Hyper-arid',2:'Arid',3:'Semi-arid',4:'Sub-humid',5:'Humid',6:'Per-humid'}
    ax3.legend(handles=[mpatches.Patch(color=_mc[int(v)-1],label=_ml.get(int(v),'?')) for v in np.unique(moisture_map) if not np.isnan(v)], loc='lower left', fontsize=7, framealpha=0.8)
    ax3.set_title('Moisture Zone', fontsize=11); ax3.set_xlabel('Longitude'); ax3.set_ylabel('Latitude'); ax3.grid(alpha=0.3)

    ax4 = fig.add_subplot(2,2,4)
    _ap = precip_clim.sum(axis=0) if (precip_clim is not None and precip_clim.ndim==3) else np.full(lgp_map.shape, precip_monthly_1d.sum())
    im4 = ax4.imshow(_ap, extent=_extent, cmap=plt.cm.YlGnBu, norm=mcolors.Normalize(vmin=0,vmax=max(_ap.max(),1000)), origin='upper', aspect='auto')
    plt.colorbar(im4, ax=ax4, label='mm/year', shrink=0.8)
    ax4.set_title(f'Annual Precipitation ({PRECIP_SOURCE})', fontsize=11); ax4.set_xlabel('Longitude'); ax4.set_ylabel('Latitude'); ax4.grid(alpha=0.3)

    plt.tight_layout(rect=[0,0,1,0.97])
    _p = os.path.join(RESULTS_DIR, f'FA4_AEZ_Maps_{COUNTRY}.png')
    plt.savefig(_p, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\u2705 Maps saved: {_p}')

In [ ]:
# @title ### 7) Crop suitability assessment {"display-mode":"form"}

CROP_REQUIREMENTS = {
    'Maize':    {'t_min':15,'t_max':35,'lgp_min':90, 'p_min':500},
    'Wheat':    {'t_min':5, 't_max':25,'lgp_min':90, 'p_min':300},
    'Sorghum':  {'t_min':18,'t_max':40,'lgp_min':75, 'p_min':250},
    'Rice':     {'t_min':20,'t_max':38,'lgp_min':120,'p_min':800},
    'Cassava':  {'t_min':18,'t_max':38,'lgp_min':180,'p_min':500},
    'Groundnut':{'t_min':22,'t_max':38,'lgp_min':90, 'p_min':400},
    'Sugarcane':{'t_min':20,'t_max':38,'lgp_min':270,'p_min':1200},
    'Coffee':   {'t_min':17,'t_max':28,'lgp_min':180,'p_min':1000},
    'Teff':     {'t_min':10,'t_max':27,'lgp_min':75, 'p_min':300},
    'Sesame':   {'t_min':25,'t_max':40,'lgp_min':75, 'p_min':300},
    'Millet':   {'t_min':20,'t_max':40,'lgp_min':60, 'p_min':200},
    'Barley':   {'t_min':5, 't_max':25,'lgp_min':60, 'p_min':250},
}

tavg_mean = tavg_monthly_1d.mean()
print(f'\nCROP SUITABILITY \u2014 {COUNTRY}')
print(f'  Tavg={tavg_mean:.1f}\u00b0C  LGP={lgp_value} days  Precip={precip_annual:.0f} mm\n')

rows = []
for crop, req in CROP_REQUIREMENTS.items():
    t_ok   = req['t_min'] <= tavg_mean  <= req['t_max']
    lgp_ok = lgp_value >= req['lgp_min']
    p_ok   = precip_annual >= req['p_min']
    if t_ok and lgp_ok and p_ok:   suit,icon='Suitable','\u2705'
    elif t_ok and (lgp_ok or p_ok):suit,icon='Marginal','\u26a0\ufe0f '
    else:                           suit,icon='Not suitable','\u274c'
    rows.append({'Crop':crop,'Suitability':suit,'Temp':('\u2713' if t_ok else '\u2717'),'LGP':('\u2713' if lgp_ok else '\u2717'),'Precip':('\u2713' if p_ok else '\u2717')})
    print(f'  {icon} {crop:<12} {suit}')

df_suit = pd.DataFrame(rows)
print(f'\n  Suitable : {len(df_suit[df_suit.Suitability=="Suitable"])}')
print(f'  Marginal : {len(df_suit[df_suit.Suitability=="Marginal"])}')
_p = os.path.join(RESULTS_DIR, f'FA4_CropSuitability_{COUNTRY}.csv')
df_suit.to_csv(_p, index=False)
print(f'\n\u2705 Saved: {_p}')